In [1]:
!pip install kagglehub pandas scikit-learn nltk

In [2]:
import kagglehub
import pandas as pd
import string
import os
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [3]:
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Path:", path)

print(os.listdir(path))

100%|██████████| 211k/211k [00:00<00:00, 27.4MB/s]

Extracting files...
Path: /root/.cache/kagglehub/datasets/uciml/sms-spam-collection-dataset/versions/1
['spam.csv']


In [4]:
file_path = os.path.join(path, "spam.csv")
df = pd.read_csv(file_path, encoding='latin-1')

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [6]:

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df["text"] = df["text"].apply(preprocess)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

In [9]:
vectorizer = TfidfVectorizer(stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [10]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [11]:
y_pred = model.predict(X_test_vec)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Naive Bayes Accuracy: 0.967713004484305

Classification Report:

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       965
        spam       1.00      0.76      0.86       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [12]:
def predict_text(text):
    text = preprocess(text)
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

In [13]:
print(predict_text("Congratulations! You won a prize"))
print(predict_text("Let's meet tomorrow"))

spam
ham


In [14]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

LinearSVC()

In [15]:
y_pred_svm = svm_model.predict(X_test_vec)

from sklearn.metrics import accuracy_score, classification_report

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nSVM Classification Report:\n")
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9802690582959641

SVM Classification Report:

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       965
        spam       0.97      0.88      0.92       150

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [16]:
def predict_svm(text):
    text = preprocess(text)
    vec = vectorizer.transform([text])
    return svm_model.predict(vec)[0]

In [19]:
print(predict_svm("Congratulations! You won a prize of 1 Cr"))
print(predict_svm("Let's meet tomorrow"))

spam
ham


In [18]:
nb_accuracy = accuracy_score(y_test, y_pred)
svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("Naive Bayes Accuracy:", nb_accuracy)
print("SVM Accuracy:", svm_accuracy)

if svm_accuracy > nb_accuracy:
    print("SVM performs better")
else:
    print("Naive Bayes performs better")

Naive Bayes Accuracy: 0.967713004484305
SVM Accuracy: 0.9802690582959641
SVM performs better
